In [ ]:
%%time

import folium
import numpy as np

In [ ]:
%%time 

# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2
# add ../funcs to the current path
import sys, os
sys.path.append(os.path.join(os.getcwd(), "../funcs")) 

In [ ]:
%%time

# read data from netcdf files, save them to 'dataset'
from jdiag import load_jdiag, get_jdiag_metadata, get_jdiag_data
dataset = load_jdiag("../data/mpasjedi/jdiag_aircar_t133.nc")

data = get_jdiag_data(dataset, "airTemperature")

In [ ]:
%%time

from base import query_data
query_data(data)

In [ ]:
%time

latitudes = data["latitude"]
longitudes = data["longitude"]
TempOmbg = data ["ombg"]
TempObs = data["ObsValue"]
TempOman = data ["oman"]
stationIDs = data["stationIdentification"]
height = data["height"]
TempObs = TempObs - 273.15


In [ ]:
def get_color(TempObs):
    if TempObs < -30:
        return "#08306B"  # dark blue
    elif TempObs < -20:
        return "#4292C6"  # light blue
    elif TempObs < -10:
        return "#7FCDBB"  # cyan/teal
    elif TempObs < 10:
        return "#41AB5D"  # green
    elif TempObs < 20:
        return "#FEE391"  # yellow
    elif TempObs < 25:
        return "#F16913"  # orange
    else:
        return "#A50F15"  # red


In [ ]:
%%time

m = folium.Map(location=[40.29, 261.05], zoom_start=4)

In [ ]:
%%time

for lat, lon, tempObs, TempOmbg, TempOman, stationID, height  in zip(latitudes, longitudes, TempObs, TempOmbg, TempOman, stationIDs, height):
    if np.ma.is_masked(tempObs) or np.ma.is_masked(TempOmbg) or np.ma.is_masked(TempOman) or np.ma.is_masked(lat) or np.ma.is_masked(lon):  
        continue  # skip this row if temp is masked   \
    
    popup_text = f"""
        <b>TempObs:</b> {float(tempObs):.2f} °C<br>
        <b>TempOmbg</b> {float(TempOmbg) :.2f} °C<br>
        <b>TempOman</b> {float(TempOman) :.2f} °C<br>
        <b>Latitude</b> {float(lat):.2f} °<br>
        <b>Longitude</b> {float(lon): .2f} °<br>
        <b>Altitude</b> {float(height):.2f} <br>
        <b>Station ID:</b> {stationID}
    """
    color = get_color(float(tempObs))  # Get color based on temperature

    folium.CircleMarker(
        location=[lat, lon],
        radius= 1,
        popup=folium.Popup(popup_text, max_width=250),
        color= color,
        fill=True,
        fill_color= color
    ).add_to(m)

In [ ]:
m